<a href="https://colab.research.google.com/github/group6aiml-hub/NIDS_AInML/blob/notebooks/Valentino_ExtraModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#1st cell to run, just importing data
import os

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q imbalanced-learn

if not os.path.exists('/content/NIDS_AInML'):
    !git clone -q https://github.com/group6aiml-hub/NIDS_AInML.git
else:
    print("Repository already cloned.")

import sys
if '/content/NIDS_AInML/src' not in sys.path:
    sys.path.append('/content/NIDS_AInML/src')

from data import last_data

d = last_data()
print("Data uploaded successfully!")

Mounted at /content/drive
Data uploaded successfully!


In [2]:
#2nd cell to run, implementing One-Hot encoding to convert strings into numbers
import pandas as pd

# 1. Let's temporarily merge X_tr and X_val to apply encoding consistently
df_tot = pd.concat([d['X_tr'], d['X_val']], axis=0)

# 2. We apply One-Hot Encoding on the categorical columns found
df_tot_enc = pd.get_dummies(df_tot, columns=['proto', 'service', 'state'], drop_first=True)

# 3. Separate the datasets again and update the dictionary 'd'
train_len = len(d['X_tr'])
d['X_tr'] = df_tot_enc.iloc[:train_len]
d['X_val'] = df_tot_enc.iloc[train_len:]

print("Conversion complete! Now the data is ready.")

Conversion complete! Now the data is ready.


In [3]:
#3rd cell to run, focusing on the binary task with a Decision Tree Classifier.
#Note that here the simple Decision Tree gets a very high score (Macro-F1: 0.9130) because distinguishing normal traffic from a generic attack is much easier.

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, f1_score
import time

# 1. Initializing the model (Decision Tree)
model = DecisionTreeClassifier(random_state=42)

# 2. Training on the BINARY task (measuring time)
start_time = time.time()
model.fit(d['X_tr'], d['y_tr_bin'])
train_time = time.time() - start_time

# 3. Prediction and evaluation on the validation set
y_pred = model.predict(d['X_val'])
macro_f1 = f1_score(d['y_val_bin'], y_pred, average='macro')

print(f"--- BINARY TASK RESULTS ---")
print(f"Training time: {train_time:.2f} seconds")
print(f"Macro-F1 (Validation): {macro_f1:.4f}\n")
print("Classification Report:")
print(classification_report(d['y_val_bin'], y_pred))

--- BINARY TASK RESULTS ---
Training time: 4.48 seconds
Macro-F1 (Validation): 0.9130

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91     10378
           1       0.92      0.92      0.92     11170

    accuracy                           0.91     21548
   macro avg       0.91      0.91      0.91     21548
weighted avg       0.91      0.91      0.91     21548



In [4]:
#Cell running the analysis through a Decision Tree Classifier to track our process.
#We started from a single tree as the initial baseline (getting 0.56 on the multi-class),
#analyzed the limitations due to imbalance and, to overcome them, introduced an advanced ensemble model (Gradient Boosting) combined with oversampling techniques (SMOTE).
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, f1_score

# We initialize the model for the multi-class
model_mc = DecisionTreeClassifier(random_state=42)
model_mc.fit(d['X_tr'], d['y_tr_mc'])

# Prediction on the multi-class validation set
y_pred_mc = model_mc.predict(d['X_val'])
macro_f1_mc = f1_score(d['y_val_mc'], y_pred_mc, average='macro')

print(f"--- RESULTS MULTI-CLASS TASKS ---")
print(f"Macro-F1 (Validation): {macro_f1_mc:.4f}\n")
print(classification_report(d['y_val_mc'], y_pred_mc))

--- RESULTS MULTI-CLASS TASKS ---
Macro-F1 (Validation): 0.5677

              precision    recall  f1-score   support

           0       0.02      0.06      0.03       319
           1       0.11      0.17      0.14       307
           2       0.29      0.27      0.28       761
           3       0.81      0.78      0.80      3969
           4       0.71      0.65      0.68      3230
           5       0.90      0.84      0.87       836
           6       0.91      0.91      0.91     10378
           7       0.86      0.72      0.79      1504
           8       0.58      0.59      0.59       218
           9       0.55      0.65      0.60        26

    accuracy                           0.78     21548
   macro avg       0.57      0.57      0.57     21548
weighted avg       0.81      0.78      0.80     21548



In [5]:
#4th cell to run, implementing SMOTE technique: since some classes contain a very small amount of support (samples),
#the SMOTE technique undertakes to Take rare class points in geometric space, look at their "neighbors,"
#and draw virtual lines in space, creating new plausible data that resembles real data but is not identical.
#This balances the dataset and the model learns to recognize the characteristics of the rare attack even in slightly different variants.
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score
import time

print("Original training set size (multi-class):", d['X_tr'].shape)

# 1. We initialize SMOTE
smote = SMOTE(random_state=42)

# 2. We apply SMOTE only on the training set
X_tr_resampled, y_tr_resampled = smote.fit_resample(d['X_tr'], d['y_tr_mc'])

print("Training set size AFTER SMOTE:", X_tr_resampled.shape)

# 3. Training Gradient Boosting on Balanced Data
print("\nStarting Gradient Boosting Training with SMOTE...")
start_time = time.time()

gb_model_smote = HistGradientBoostingClassifier(random_state=42)
gb_model_smote.fit(X_tr_resampled, y_tr_resampled)

train_time = time.time() - start_time

# 4. Evaluation on the ORIGINAL validation set
y_pred_gb_smote = gb_model_smote.predict(d['X_val'])
macro_f1_gb_smote = f1_score(d['y_val_mc'], y_pred_gb_smote, average='macro')

print(f"\n--- GRADIENT BOOSTING RESULTS WITH SMOTE (MULTI-CLASS) ---")
print(f"Training time: {train_time:.2f} seconds")
print(f"Macro-F1 (Validation): {macro_f1_gb_smote:.4f}\n")
print("Classification Report:")
print(classification_report(d['y_val_mc'], y_pred_gb_smote))

Original training set size (multi-class): (86192, 56)
Training set size AFTER SMOTE: (415120, 56)

Starting Gradient Boosting Training with SMOTE...

--- GRADIENT BOOSTING RESULTS WITH SMOTE (MULTI-CLASS) ---
Training time: 91.80 seconds
Macro-F1 (Validation): 0.5751

Classification Report:
              precision    recall  f1-score   support

           0       0.16      0.52      0.24       319
           1       0.18      0.44      0.26       307
           2       0.41      0.27      0.33       761
           3       0.88      0.72      0.79      3969
           4       0.66      0.82      0.73      3230
           5       0.94      0.84      0.89       836
           6       0.97      0.86      0.91     10378
           7       0.84      0.77      0.80      1504
           8       0.42      0.92      0.57       218
           9       0.14      0.77      0.24        26

    accuracy                           0.79     21548
   macro avg       0.56      0.69      0.58     21548
weig

In [6]:
#While Random Forest creates lots of trees in parallel and averages the grades, Gradient Boosting works sequentially.
#The first tree makes an initial estimate of network traffic and inevitably makes mistakes (it fails to classify some attacks).
#The second tree does not start from scratch: it focuses exclusively on the errors (residuals) made by the first tree, trying to correct them.
#The third tree corrects the remaining errors of the second, and so on, one step at a time (via a mathematical technique similar to gradient descent).
#The final result is the sum of the contributions of all the trees put together.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score
import time

# 1. We initialize Gradient Boosting optimized for complex datasets
print("Starting Gradient Boosting training...")
start_time = time.time()

gb_model = HistGradientBoostingClassifier(random_state=42)

# 2. Multi-class task training
gb_model.fit(d['X_tr'], d['y_tr_mc'])

train_time = time.time() - start_time

# 3. Prediction and evaluation on the validation set
y_pred_gb = gb_model.predict(d['X_val'])
macro_f1_gb = f1_score(d['y_val_mc'], y_pred_gb, average='macro')

print(f"\n--- GRADIENT BOOSTING RESULTS (MULTI-CLASS) ---")
print(f"Training time: {train_time:.2f} seconds")
print(f"Macro-F1 (Validation): {macro_f1_gb:.4f}\n")
print("Classification Report:")
print(classification_report(d['y_val_mc'], y_pred_gb))

Starting Gradient Boosting training...

--- GRADIENT BOOSTING RESULTS (MULTI-CLASS) ---
Training time: 13.55 seconds
Macro-F1 (Validation): 0.5676

Classification Report:
              precision    recall  f1-score   support

           0       0.19      0.29      0.23       319
           1       0.31      0.20      0.24       307
           2       0.58      0.14      0.23       761
           3       0.81      0.84      0.82      3969
           4       0.60      0.80      0.69      3230
           5       0.98      0.83      0.90       836
           6       0.95      0.90      0.92     10378
           7       0.88      0.74      0.81      1504
           8       0.62      0.71      0.66       218
           9       0.11      0.58      0.18        26

    accuracy                           0.81     21548
   macro avg       0.60      0.60      0.57     21548
weighted avg       0.83      0.81      0.81     21548



RESULTS ANALYSIS:

Although the final Macro-F1 score is identical between the two models (0.5677 vs 0.5676), looking within the individual classes known that the two models performed differently:
  1)The Decision Tree tended to make "dry" decisions based on training data; on very rare classes (e.g. class 0 or class 9), it obtained an almost zero F1-score (0.03 and 0.60 respectively).
  2)The Gradient Boosting worked better on the middle/small classes thanks to sequential training: on class 0, for example, the F1-score rose from 0.03 to 0.23; on class 1, the F1-score rose from 0.14 to 0.24. Global Accuracy even rose from 78% to 81%.
  3)How is it possible that the final Macro-F1 remained at 0.56? While Gradient Boosting improved many rare classes, class 9 (which has just 26 examples out of over 21,000) dropped from 0.60 to 0.18! Since Macro-F1 is a pure mean across the 10 classes, this sharp decline on class 9 offset the gains made on the other classes, keeping the arithmetic mean almost identical.

Increasing model complexity (from a single tree to a Gradient Boosting ensemble) improves overall accuracy and sensitivity across many classes, but it alone is not enough to solve the problem if the dataset is severely unbalanced. Classes with very few samples will continue to suffer if we do not apply an explicit balancing technique like SMOTE.